# AIC25 — Tier-2: Proper 3D-HOTA Numbers

**Goal:** produce the official **3D-HOTA** score for Warehouse_016 — the defensible, paper-comparable metric (Glance-MCMT paper scored HOTA 43–51).

This is the **heavy** path. It needs, in order:
1. **Depth maps** (⚠️ 30–80 GB download) — required for 3D world coordinates.
2. **All cameras** — multi-camera HOTA associates IDs across every view, so this is **not** a few-camera subset.
3. Single-camera tracking **re-run with depth present** (the Tier-1 JSONs skipped depth → no world coords → unusable here).
4. Multi-camera association → multi-camera fix → TrackEval HOTA.

> For best numbers, also train the AIC25 detector (one-time ~6–10h). Without it the ByteTrack fallback misses forklifts/robots and HOTA will be low.

Branch: **`hithesh/combined-pipeline`**. Run on a **T4 GPU** runtime. Reuses the same Steps 0–5 as the Tier-1 notebook; the new parts are **Step 6 (depth)**, **[G]**, **[H]**.

---
## Step 0 — Environment + Drive

In [ ]:
import os, sys
ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if ON_COLAB:
    REPO='/content/repo'; PY='python'; DRIVE='/content/drive/MyDrive/AIC25'
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'): drive.mount('/content/drive')
    else: print('Drive already mounted.')
    for d in ['models','outputs/Detection','outputs/EmbedFeature','outputs/Tracking']:
        os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    print('Colab | Drive:', DRIVE)
else:
    REPO='/home/seco/deepLearning/Single-Camera-Tracking-Consistency'; PY=f'{REPO}/.venv/bin/python'; DRIVE=None
    os.chdir(REPO); print('Local')
print('REPO:', REPO)

---
## Step 1 — Clone + checkout combined branch + install

In [ ]:
if ON_COLAB:
    import subprocess as _sp
    if not os.path.exists(REPO):
        os.system(f'git clone https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git {REPO}')
    else:
        os.system(f'git -C {REPO} fetch --quiet')
    os.chdir(REPO)
    rc = os.system(f'git -C {REPO} checkout hithesh/combined-pipeline')
    os.system(f'git -C {REPO} pull --quiet 2>/dev/null')
    if rc != 0 or not os.path.isdir(f'{REPO}/tracklet_repair'):
        raise RuntimeError('Checkout failed or tracklet_repair missing — push: git push -u origin hithesh/combined-pipeline')
    print('On hithesh/combined-pipeline ✓')
    for _p in ['thop','loguru','lap','motmetrics','filterpy','easydict','yacs','termcolor',
               'prettytable','tabulate','ninja','cython_bbox','pycocotools','h5py']:
        _sp.run(['pip','install','-q',_p], capture_output=True, text=True)
    if _sp.run(['pip','install','-q','faiss-gpu'], capture_output=True).returncode != 0:
        _sp.run(['pip','install','-q','faiss-cpu'], capture_output=True)
    os.chdir(f'{REPO}/BoT-SORT');         os.system('python setup.py develop --quiet 2>/dev/null')
    os.chdir(f'{REPO}/deep-person-reid'); os.system('python setup.py develop --quiet 2>/dev/null')
    os.chdir(REPO); os.system('pip install -q -r tracking/requirements.txt 2>/dev/null')
    print('Dependencies installed.')
else:
    print('Local: skip.')

---
## Step 2 — GPU check

In [ ]:
import subprocess
r = subprocess.run([PY,'-c','import torch; print("CUDA:", torch.cuda.is_available())'], capture_output=True, text=True)
print(r.stdout.strip())
if ON_COLAB and 'CUDA: False' in r.stdout:
    raise RuntimeError('NO GPU — Runtime → Change runtime type → T4 GPU, Restart, re-run.')

---
## Step 3 — Models (OSNet + ByteTrack; AIC25 detector if trained)

In [ ]:
if ON_COLAB:
    os.system('pip install -q gdown'); M=f'{DRIVE}/models'
    def get_model(local, on_drive, gid, name):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        if os.path.exists(local): print(f'{name}: local'); return
        if os.path.exists(on_drive): os.system(f'cp "{on_drive}" "{local}"'); print(f'{name}: from Drive'); return
        for a in range(3):
            os.system(f'gdown "https://drive.google.com/uc?id={gid}" -O "{on_drive}"')
            if os.path.exists(on_drive) and os.path.getsize(on_drive)>100000: break
            print(f'  retry {a+1}/3')
        if os.path.exists(on_drive): os.system(f'cp "{on_drive}" "{local}"'); print(f'{name}: done')
        else: print(f'{name}: MISSING — download manually to {local}')
    get_model(f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar', f'{M}/osnet_ms_m_c.pth.tar','1IosIFlLiulGIjwW3H8uMCC3YvMyr9gZ2','OSNet')
    get_model(f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar', f'{M}/bytetrack_x_mot17.pth.tar','1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5','ByteTrack')
    aic = f'{M}/ai_city_ckpt.pth.tar'
    if os.path.exists(aic): os.system(f'cp "{aic}" "{REPO}/BoT-SORT/ai_city_ckpt.pth.tar"'); print('AIC25 detector: from Drive ✓')
    else: print('AIC25 detector: not trained — ByteTrack fallback (HOTA will be lower)')
else: print('Local: models in place.')

---
## Step 4 — Scene config
Multi-camera HOTA uses **all** cameras — leave CAMERAS = None.

In [ ]:
SCENE='Warehouse_016'; DATASET='Val'; CAMERAS=None   # None = all cameras (required for multi-cam HOTA)
os.chdir(REPO); print('Scene:', SCENE, DATASET, '| cameras: ALL')

---
## Step 5 — Download videos + ground_truth (HuggingFace)

In [ ]:
if ON_COLAB:
    import shutil, getpass
    os.system('pip install -q huggingface_hub'); from huggingface_hub import snapshot_download, login
    dd=f'{DRIVE}/datasets/{DATASET}/{SCENE}'; ld=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    dv=f'{dd}/videos'; vl=f'{ld}/videos'
    os.makedirs(dd, exist_ok=True); os.makedirs(ld, exist_ok=True)
    if os.path.exists(dv) and os.listdir(dv):
        print('[CACHE HIT] videos on Drive.')
    else:
        tok=None
        try:
            from google.colab import userdata; tok=userdata.get('HF_TOKEN')
        except Exception: pass
        if not tok: tok=getpass.getpass('HF token: ')
        login(token=tok); sp=DATASET.lower()
        print('Downloading videos + GT...', flush=True)
        snapshot_download('nvidia/PhysicalAI-SmartSpaces', repo_type='dataset', local_dir='/content/hf_tmp',
            allow_patterns=[f'MTMC_Tracking_2025/{sp}/{SCENE}/videos/**',
                            f'MTMC_Tracking_2025/{sp}/{SCENE}/calibration.json',
                            f'MTMC_Tracking_2025/{sp}/{SCENE}/ground_truth.json'])
        src=f'/content/hf_tmp/MTMC_Tracking_2025/{sp}/{SCENE}'
        if not os.path.exists(dv): shutil.copytree(f'{src}/videos', dv)
        for fn in ['calibration.json','ground_truth.json']:
            if os.path.exists(f'{src}/{fn}'): shutil.copy(f'{src}/{fn}', f'{dd}/{fn}')
        shutil.rmtree('/content/hf_tmp', ignore_errors=True)
    for fn in ['calibration.json','ground_truth.json']:
        s,d=f'{dd}/{fn}',f'{ld}/{fn}'
        if os.path.exists(s) and not os.path.exists(d): shutil.copy(s,d)
    if os.path.islink(vl): os.unlink(vl)
    os.makedirs(vl, exist_ok=True)
    cams=sorted(os.path.splitext(f)[0] for f in os.listdir(dv) if f.endswith('.mp4'))
    print(f'✓ {len(cams)} cameras: {cams}')
else: print('Local: existing data.')

---
## Step 6 — Download DEPTH MAPS ⚠️ (30–80 GB)
Required for 3D world coordinates. HuggingFace stores them as `depth_maps/` (plural); the code expects `depth_map/<Camera>.h5` (singular). This cell downloads and remaps. Cached to Drive — only the first session pays the cost.

In [ ]:
if ON_COLAB:
    import shutil, glob
    os.system('pip install -q huggingface_hub'); from huggingface_hub import snapshot_download
    sp=DATASET.lower()
    dd=f'{DRIVE}/datasets/{DATASET}/{SCENE}'; ld=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    drive_depth=f'{dd}/depth_map'; local_depth=f'{ld}/depth_map'
    os.makedirs(drive_depth, exist_ok=True)
    if os.path.exists(drive_depth) and any(f.endswith('.h5') for f in os.listdir(drive_depth)):
        print('[CACHE HIT] depth maps on Drive:', len([f for f in os.listdir(drive_depth) if f.endswith('.h5')]), 'files')
    else:
        print('Downloading depth maps (30-80 GB) — slow, first time only...', flush=True)
        snapshot_download('nvidia/PhysicalAI-SmartSpaces', repo_type='dataset', local_dir='/content/hf_depth',
            allow_patterns=[f'MTMC_Tracking_2025/{sp}/{SCENE}/depth_map*/**'])
        # collect any .h5 found and place as depth_map/<Camera>.h5 on Drive
        found=glob.glob(f'/content/hf_depth/MTMC_Tracking_2025/{sp}/{SCENE}/depth_map*/**/*.h5', recursive=True)             + glob.glob(f'/content/hf_depth/MTMC_Tracking_2025/{sp}/{SCENE}/depth_map*/*.h5')
        print(f'  found {len(found)} .h5 files')
        for p in found: shutil.copy(p, f'{drive_depth}/{os.path.basename(p)}')
        shutil.rmtree('/content/hf_depth', ignore_errors=True)
        print('  depth maps cached to Drive.')
    # link Drive depth → local repo path the code reads
    if os.path.islink(local_depth) or os.path.isdir(local_depth):
        if not os.path.islink(local_depth): shutil.rmtree(local_depth, ignore_errors=True)
        if os.path.islink(local_depth): os.unlink(local_depth)
    os.symlink(drive_depth, local_depth)
    h5s=[f for f in os.listdir(local_depth) if f.endswith('.h5')]
    print(f'✓ depth_map/ ready — {len(h5s)} files, e.g. {h5s[:3]}')
    if not h5s:
        print('⚠ NO .h5 found — inspect the HF folder names; the allow_patterns may need adjusting.')
else: print('Local: depth maps expected under AIC25_Track1/.../depth_map/')

---
## Link outputs to Drive

In [ ]:
if ON_COLAB:
    import shutil as _s
    for folder in ['Detection','EmbedFeature','Tracking']:
        dfo=f'{DRIVE}/outputs/{folder}'; rfo=f'{REPO}/{folder}'; os.makedirs(dfo, exist_ok=True)
        if os.path.islink(rfo): pass
        elif os.path.isdir(rfo): _s.move(rfo,dfo); os.symlink(dfo,rfo)
        else: os.symlink(dfo,rfo)
    print('Outputs linked to Drive.')
else: print('Local.')

---
## Generate single-camera JSONs **with depth** (all cameras)
Same detect→embed→track→fix chain as Tier-1, but now depth_map/ is present so single-camera tracking computes **3D world coordinates** (needed downstream). Frames kept until tracking done.

In [ ]:
import os, shutil, cv2, subprocess
os.chdir(REPO)
_dv=f'{DRIVE}/datasets/{DATASET}/{SCENE}/videos' if DRIVE else None
_lv=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos'
cam_source=_dv if (_dv and os.path.exists(_dv)) else _lv
_items=os.listdir(cam_source)
_mp4s=sorted(os.path.splitext(f)[0] for f in _items if f.endswith('.mp4'))
_dirs=sorted(c for c in _items if os.path.isdir(f'{cam_source}/{c}') and 'map' not in c)
all_cams=_mp4s if _mp4s else _dirs
cams=[c for c in CAMERAS if c in all_cams] if CAMERAS else all_cams
print('Cameras:', cams)
if os.path.exists(f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'):
    ckpt='BoT-SORT/ai_city_ckpt.pth.tar'; exp_file='BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py'; print('Detector: AIC25')
else:
    ckpt='BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; exp_file='BoT-SORT/yolox/exps/example/mot/yolox_x_mix_det.py'; print('Detector: ByteTrack fallback')

def extract_frames(cam):
    fd=f'{_lv}/{cam}/Frame'
    if os.path.exists(fd) and os.listdir(fd): return fd
    mp4=f'{cam_source}/{cam}.mp4'
    if not os.path.exists(mp4):
        d=f'{cam_source}/{cam}'; mp4=next((f'{d}/{f}' for f in os.listdir(d) if f.endswith('.mp4')),None) if os.path.isdir(d) else None
    if not mp4: print('  no mp4', cam); return None
    os.makedirs(fd, exist_ok=True); cap=cv2.VideoCapture(mp4); n=1; ok,frm=cap.read()
    while ok:
        cv2.imwrite(f'{fd}/{str(n).zfill(6)}.jpg', frm); ok,frm=cap.read()
        if n%2000==0: print(f'    {cam}: {n}', flush=True)
        n+=1
    cap.release(); print(f'  {cam}: {n-1} frames'); return fd
def run(cmd):
    r=subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode!=0: print('\n'.join(r.stdout.strip().splitlines()[-40:]))
    return r.returncode

for cam in cams:
    print(f'\n=== [B+C] {cam} ===')
    if extract_frames(cam) is None: continue
    if os.path.exists(f'{REPO}/Detection/{SCENE}/{cam}.txt'): print('  det exists'); continue
    run(f'{PY} BoT-SORT/tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} --camera {cam} -f {exp_file} -c {ckpt} ./')
os.chdir(REPO)
det_dir=f'{REPO}/Detection/{SCENE}'
for cam in [os.path.splitext(f)[0] for f in os.listdir(det_dir) if f.endswith('.txt')] if os.path.isdir(det_dir) else []:
    fd=f'{_lv}/{cam}/Frame'
    if not (os.path.exists(fd) and os.listdir(fd)): extract_frames(cam)
print('\n=== [D] embeddings ===')
os.chdir(f'{REPO}/deep-person-reid'); print('  rc', os.system(f'{PY} torchreid/aic25_extract.py -s {SCENE} --dataset {DATASET} ../')); os.chdir(REPO)
for cam in cams:
    print(f'\n=== [E+F] {cam} ===')
    run(f'{PY} BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET}')
    run(f'{PY} BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET}')
for cam in cams: shutil.rmtree(f'{_lv}/{cam}', ignore_errors=True)
print('\n[DONE] single-camera JSONs (with world coords) ready.')

---
## [G] Multi-camera tracking + fix
`multi_camera_revised.py` **auto-creates** an `expN` dir (`get_next_experiment_id`). We detect that dir and pass it to `multi_camera_fix.py --exp_path` so the paths line up (the old notebook hard-coded mismatched exp names).

In [ ]:
import os
os.chdir(REPO)
mc_dir=f'{REPO}/Tracking/Multicamera/{SCENE}'
before=set(os.listdir(mc_dir)) if os.path.isdir(mc_dir) else set()
print('[G] multi_camera_revised...')
rc=os.system(f'{PY} BoT-SORT/multi_camera_revised.py -s {SCENE} --dataset {DATASET}')
print('  revised rc', rc)
after=set(os.listdir(mc_dir)) if os.path.isdir(mc_dir) else set()
new=sorted(after-before)
EXP=new[-1] if new else (sorted(after)[-1] if after else 'exp1')
print('  multi-camera exp dir =', EXP)
print('  contents:', os.listdir(f'{mc_dir}/{EXP}') if os.path.isdir(f'{mc_dir}/{EXP}') else 'MISSING')
print('[G] multi_camera_fix...')
rc=os.system(f'{PY} BoT-SORT/multi_camera_fix.py -s {SCENE} --dataset {DATASET} --exp_path {EXP}')
print('  fix rc', rc)
# what files exist now (helps debug the eval step)
ed=f'{mc_dir}/{EXP}'
print('  exp dir files:', os.listdir(ed) if os.path.isdir(ed) else 'MISSING')
for sub in ['', 'output_result']:
    p=os.path.join(ed, sub)
    if os.path.isdir(p): print(f'   {sub or "."}/:', [f for f in os.listdir(p)])

---
## [H] Evaluate — 3D HOTA
`prepare_eval_data.py` reads `Tracking/Multicamera/<scene>/<EXP>/fixed_whole_tracking_results.json`. We pass the **same EXP** detected in [G]. If `fixed_whole_tracking_results.json` isn't found, the [G] diagnostics above show what `multi_camera_fix.py` actually produced — adjust the filename/exp accordingly.

In [ ]:
import os
os.chdir(f'{REPO}/TrackEval')
print('[H] prepare_eval_data (EXP=%s)...' % EXP)
rc=os.system(f'{PY} prepare_eval_data.py -s {SCENE} --exp {EXP} --dataset {DATASET} --base_dir {REPO}')
print('  prepare rc', rc)
gt_txt=f'{REPO}/TrackEval/aicity_25_data/{SCENE}/{EXP}.txt'
print('  tracker txt exists:', os.path.exists(gt_txt))
print('[H] main (HOTA)...')
rc=os.system(f'{PY} main.py -s {SCENE} --exp {EXP}')
print('  eval rc', rc, '\n--- look above for HOTA / DetA / AssA / LocA ---')

---
## Before/after-repair HOTA *(follow-up, not automated here)*
To show Hakan's `tracklet_repair` improves HOTA, re-run **[G]+[H]** using single-camera JSONs that have been passed through `tracklet_repair` **before** multi-camera. That requires writing the repaired output back into the project JSON schema (with world coords + features) that `multi_camera_revised.py` consumes — an integration step, not a one-liner. Do the official baseline HOTA above first; wire the repaired-input variant once it's confirmed working.